# This code calibrates and evaluate the Stochastic model in the case of output-driven diffusion. It can be adapted to other diffusions

# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize # USE IN THE MODEL CALIBRATION

from google.colab import files
import zipfile
import os

##CAMELS-DATA from Python Library.

In [ ]:
pip install aqua-fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.1/289.1 kB 6.6 MB/s eta 0:00:00


In [ ]:
from aqua_fetch import RainfallRunoff

# Initialiser directement avec le nom du dataset
rr = RainfallRunoff("CAMELS_FR")

/usr/local/lib/python3.12/dist-packages/aqua_fetch/rr/utils.py:127: UserWarning: netCDF4 module is not installed. Please install it to save data in netcdf format
  warnings.warn(msg, UserWarning)


downloading 6 files to /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
downloading ADDITIONAL_LICENSES.zip
0% of 0.07 MB downloaded
100% of 0.07 MB downloaded
downloading CAMELS_FR_attributes.zip
0% of 9.88 MB downloaded
100% of 9.88 MB downloaded
downloading CAMELS_FR_geography.zip
0% of 1.45 MB downloaded
100% of 1.45 MB downloaded
downloading CAMELS_FR_time_series.zip
0% of 361.39 MB downloaded
20% of 361.39 MB downloaded
40% of 361.39 MB downloaded
60% of 361.39 MB downloaded
80% of 361.39 MB downloaded
100% of 361.39 MB downloaded
downloading README.md
0% of 0.01 MB downloaded
100% of 0.01 MB downloaded
downloading CAMELS-FR_description.ods
0% of 0.05 MB downloaded
100% of 0.05 MB downloaded
unzipping files in /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
unzipping CAMELS_FR_attributes.zip to CAMELS_FR_attributes
unzipping CAMELS_FR_geography.zip to CAMELS_FR_geography
unzipping ADDITIONAL_LICENSES.zip to ADDITIONAL_LICENSES
unzi

In [ ]:
meta, ds = rr.fetch()

Read 654 stations for 22 dyn features in 48.09 seconds with 2 cpus.


In [ ]:
import xarray as xr

print(ds)              # aperçu global
#print(ds.coords)       # coordonnées (temps, variables, stations)
#print(ds['912101A'].head())  # exemple : station 5

<xarray.Dataset> Size: 2GB
Dimensions:           (time: 18993, dynamic_features: 22)
Coordinates:
  * time              (time) datetime64[ns] 152kB 1970-01-01 ... 2021-12-31
  * dynamic_features  (dynamic_features) object 176B 'q_cms_obs' ... 'airtemp...
Data variables: (12/654)
    A105003001        (time, dynamic_features) float64 3MB 1.65e+03 ... 13.4
    A107020001        (time, dynamic_features) float64 3MB nan nan ... 4.1 13.3
    A112020001        (time, dynamic_features) float64 3MB 1.04e+03 ... 13.4
    A116003002        (time, dynamic_features) float64 3MB nan nan ... 4.3 13.3
    A140202001        (time, dynamic_features) float64 3MB nan nan ... 6.8 12.4
    A202030001        (time, dynamic_features) float64 3MB nan nan ... 7.1 13.2
    ...                ...
    Y661401001        (time, dynamic_features) float64 3MB 860.0 0.441 ... 14.3
    Y781000101        (time, dynamic_features) float64 3MB nan nan ... 6.4 15.8
    Y862000101        (time, dynamic_features) float64 3MB 

In [ ]:
print(ds.dynamic_features.values)

['q_cms_obs' 'q_mm_obs' 'tsd_val_s' 'tsd_val_q' 'tsd_val_m' 'tsd_val_c'
 'tsd_val_i' 'pcp_mm' 'pcp_mm_solfrac' 'airtemp_C_mean' 'pet_mm_ou'
 'pet_mm_pe' 'pet_mm_pm' 'windspeed_mps' 'spechum_gkg' 'lwdownrad_wm2'
 'solrad_wm2' 'tsd_swi_gr' 'tsd_swi_isba' 'tsd_swe_isba' 'airtemp_C_min'
 'airtemp_C_max']


In [ ]:
# Full period
print(ds["time"].values[0], ds["time"].values[-1])

1970-01-01T00:00:00.000000000 2021-12-31T00:00:00.000000000


## Period used

In [ ]:
#Period used
ds_recent = ds.sel(time=slice("2000-01-01", "2021-12-31"))

# Liste de toutes les stations
all_stations = list(ds_recent.keys())

# Sélection des colonnes d'intérêt
features = ["q_mm_obs", "pcp_mm", "pet_mm_pm"]

# Compter les stations sans NaN
valid_stations = []

for station_id in all_stations:
    subset = ds_recent[station_id].sel(dynamic_features=features)
    df = subset.to_dataframe().reset_index().pivot(
        index="time", columns="dynamic_features", values=station_id
    )

    if df[features].isna().sum().max() == 0:  # aucune valeur manquante
        valid_stations.append(station_id)

print(f"Nombre total de stations sans données manquantes: {len(valid_stations)}")


Nombre total de stations sans données manquantes: 263


GRHyMoLAP

In [ ]:
import numpy as np
from scipy.optimize import minimize
from numba import njit

# ============================================
# NUMBA FUNCTIONS
# ============================================

@njit
def Percolation(Pn, En, X1):
    n = len(Pn)
    S = np.zeros(n)
    Perc = np.zeros(n)

    S[0] = X1 / 2.0
    ratio = (4.0 / 9.0) * (S[0] / X1)
    Perc[0] = S[0] * (1 - (1 + ratio**4) ** (-0.25))

    for i in range(1, n):
        temp = (S[i-1] / X1) ** 2

        frac = Pn[i] / X1
        Ps = X1 * (1 - temp) * np.tanh(frac) / (1 + (S[i-1] / X1) * np.tanh(frac))

        frac = En[i] / X1
        Es = S[i-1] * (2 - S[i-1]/X1) * np.tanh(frac) / (1 + (1 - S[i-1]/X1) * np.tanh(frac))

        S[i] = S[i-1] + Ps - Es

        ratio = (4.0 / 9.0) * (S[i] / X1)
        Perc[i] = S[i] * (1 - (1 + ratio**4) ** (-0.25))

        S[i] -= Perc[i]

    return Perc


@njit
def GRHyMoLAP_Model(params, Q0, Pn, En):
    MU, LAMBDA, X1, GAMMA = params
    N = len(Pn)

    Q = np.zeros(N)
    Q[0] = Q0

    Perc = Percolation(Pn, En, X1)

    for t in range(N-1):
        Q[t+1] = max(
            0.0,
            Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1)
            + GAMMA * Perc[t+1] * Pn[t+1]
        )

    return Q


@njit
def NSE(obs, sim):
    n = len(obs)
    mean_obs = 0.0
    count = 0

    for i in range(n):
        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            mean_obs += obs[i]
            count += 1

    if count == 0:
        return np.nan

    mean_obs /= count

    num = 0.0
    den = 0.0

    for i in range(n):
        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            num += (sim[i] - obs[i]) ** 2
            den += (obs[i] - mean_obs) ** 2

    if den == 0:
        return np.nan

    return 1.0 - num / den


@njit
def RMSE(obs, sim):
    n = len(obs)
    mse = 0.0
    count = 0

    for i in range(n):
        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            diff = sim[i] - obs[i]
            mse += diff * diff
            count += 1

    if count == 0:
        return np.nan

    return np.sqrt(mse / count)


# ============================================
# OBJECTIVE (minimize RMSE)
# ============================================
def objective(params, Q0, Pn_train, En_train, Q_obs_train):
    Q_sim = GRHyMoLAP_Model(np.array(params), Q0, Pn_train, En_train)
    rmse = RMSE(Q_obs_train, Q_sim)
    return rmse if np.isfinite(rmse) else 1e9


# ============================================
# General parameters
# ============================================
b1_ratio = 0.6
max_missing_ratio = 0.05

stations = all_stations
results = {}

# ============================================
# Main loop
# ============================================
i = 0
for station_id in stations:
    i += 1
    print(f"\n=== Station {station_id} ===, Number = {i}")

    Q_obs = ds_recent[station_id].sel(dynamic_features="q_mm_obs").to_numpy()
    P     = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()
    PET   = ds_recent[station_id].sel(dynamic_features="pet_mm_pm").to_numpy()

    Pn = np.maximum(0, P - PET)
    En = np.maximum(0, PET - P)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        print("⚠️ Station skipped (no valid data).")
        continue

    missing_count = np.sum(np.isnan(Q_obs))
    missing_ratio = missing_count / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too many missing values ({missing_ratio*100:.1f}%)")
        continue

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    # ============================================
    # Optimization (multi-start)
    # ============================================
    initial_guesses = [
        [1.0, 8, 150, 0.1],
        [0.6, 2, 400, 1],
        [1.4, 15, 300, 0.5],
        [1., 10, 1000, 0.3],
        [1.8, 5, 800, 0.5]
    ]

    best_res = None
    best_val = float("inf")

    for guess in initial_guesses:
        res = minimize(
            objective,
            guess,
            args=(Q0, Pn[:b1], En[:b1], Q_obs[:b1]),
            method="Nelder-Mead",
            options={'maxiter': 2500, 'disp': False}
        )

        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    MU, LAMBDA, X1, GAMMA = best_res.x
    RMSE_cal = best_res.fun
    NSE_cal = NSE(Q_obs[:b1], GRHyMoLAP_Model(best_res.x, Q0, Pn[:b1], En[:b1]))

    # ============================================
    # Full simulation
    # ============================================
    Qsim = GRHyMoLAP_Model(
        np.array([MU, LAMBDA, X1, GAMMA]),
        Q0, Pn, En
    )

    NSE_val = NSE(Q_obs[b1:], Qsim[b1:])
    RMSE_val = RMSE(Q_obs[b1:], Qsim[b1:])

    print(f"✅ Calibration RMSE: {RMSE_cal:.3f}, Validation RMSE: {RMSE_val:.3f}")
    print(f"   Calibration NSE: {NSE_cal:.3f}, Validation NSE: {NSE_val:.3f}")
    print(f"   Params: mu={MU:.3f}, lambda={LAMBDA:.3f}, X1={X1:.3f}, GAMMA={GAMMA:.3f}")

    Perc = Percolation(Pn, En, X1)

    results[station_id] = {
        "params": [MU, LAMBDA, X1, GAMMA],
        "NSE_cal": NSE_cal,
        "NSE_val": NSE_val,
        "RMSE_cal": RMSE_cal,
        "RMSE_val": RMSE_val,
        "Qsim": Qsim,
        "Perc": Perc,
        "Pn": Pn,
        "Q_obs": Q_obs,
        "missing_ratio": missing_ratio,
        "missing_count": missing_count,
    }

print(f"\n✅ Simulation completed for {len(results)} valid basins.")


=== Station A105003001 ===, Number = 1
✅ Calibration RMSE: 0.804, Validation RMSE: 0.617
   Calibration NSE: 0.625, Validation NSE: 0.645
   Params: mu=1.124, lambda=6.466, X1=421.416, GAMMA=0.129

=== Station A107020001 ===, Number = 2
✅ Calibration RMSE: 0.679, Validation RMSE: 0.612
   Calibration NSE: 0.638, Validation NSE: 0.602
   Params: mu=1.152, lambda=7.386, X1=371.219, GAMMA=0.112

=== Station A112020001 ===, Number = 3
⚠️ Too many missing values (63.5%)

=== Station A116003002 ===, Number = 4
✅ Calibration RMSE: 0.746, Validation RMSE: 0.584
   Calibration NSE: 0.667, Validation NSE: 0.704
   Params: mu=1.122, lambda=6.857, X1=310.142, GAMMA=0.136

=== Station A140202001 ===, Number = 5
✅ Calibration RMSE: 3.151, Validation RMSE: 3.527
   Calibration NSE: 0.644, Validation NSE: 0.645
   Params: mu=1.192, lambda=12.457, X1=165.421, GAMMA=0.261

=== Station A202030001 ===, Number = 6
✅ Calibration RMSE: 1.229, Validation RMSE: 1.209
   Calibration NSE: 0.791, Validation NSE:

NSE result summary

In [ ]:
# =============================================================
# 📌 EXTRACTION OF NSE & RMSE — CALIBRATION
# =============================================================
nse_cal = [res['NSE_cal'] for res in results.values() if not np.isnan(res['NSE_cal'])]
rmse_cal = [res['RMSE_cal'] for res in results.values() if not np.isnan(res['RMSE_cal'])]

print("\n================= CALIBRATION =================\n")

# ----- NSE -----
if nse_cal:
    print(f"NSE Calibration -> Median: {np.percentile(nse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_cal, 95):.4f}")
    print("MEAN NSE_CAL:", np.mean(nse_cal))
    print("MIN NSE_CAL:", np.min(nse_cal))
    print("MAX NSE_CAL:", np.max(nse_cal))
else:
    print("No NSE available for the calibration.")

# ----- RMSE -----
print("\n--- RMSE Calibration ---")
if rmse_cal:
    print(f"RMSE Calibration -> Median: {np.percentile(rmse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(rmse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(rmse_cal, 95):.4f}")
    print("MEAN RMSE_CAL:", np.mean(rmse_cal))
    print("MIN RMSE_CAL:", np.min(rmse_cal))
    print("MAX RMSE_CAL:", np.max(rmse_cal))
else:
    print("No RMSE available for the calibration.")


# =============================================================
# 📌 EXTRACTION OF NSE & RMSE — VALIDATION
# =============================================================
print("\n\n================= VALIDATION =================\n")

nse_val = [res['NSE_val'] for res in results.values() if not np.isnan(res['NSE_val'])]
rmse_val = [res['RMSE_val'] for res in results.values() if not np.isnan(res['RMSE_val'])]

# ----- NSE -----
if nse_val:
    print(f"NSE Validation -> Median: {np.percentile(nse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_val, 95):.4f}")
    print("MEAN NSE_VAL:", np.mean(nse_val))
    print("MIN NSE_VAL:", np.min(nse_val))
    print("MAX NSE_VAL:", np.max(nse_val))
else:
    print("No valid station for NSE in validation.")

# ----- RMSE -----
print("\n--- RMSE Validation ---")
if rmse_val:
    print(f"RMSE Validation -> Median: {np.percentile(rmse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(rmse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(rmse_val, 95):.4f}")
    print("MEAN RMSE_VAL:", np.mean(rmse_val))
    print("MIN RMSE_VAL:", np.min(rmse_val))
    print("MAX RMSE_VAL:", np.max(rmse_val))
else:
    print("No valid station for RMSE in validation.")


================= CALIBRATION =================

NSE Calibration -> Median: 0.752, 5th percentile: 0.5312, 95th percentile: 0.8748
MEAN NSE_CAL: 0.7348975492566159
MIN NSE_CAL: 0.082471668583884
MAX NSE_CAL: 0.9203730712875949

--- RMSE Calibration ---
RMSE Calibration -> Median: 0.610, 5th percentile: 0.1757, 95th percentile: 2.1682
MEAN RMSE_CAL: 0.8077776968028727
MIN RMSE_CAL: 0.07940625648613109
MAX RMSE_CAL: 4.989388141735578


================= VALIDATION =================

NSE Validation -> Median: 0.757, 5th percentile: 0.5009, 95th percentile: 0.8926
MEAN NSE_VAL: 0.733903780099122
MIN NSE_VAL: 0.061018959139902296
MAX NSE_VAL: 0.93815555850702

--- RMSE Validation ---
RMSE Validation -> Median: 0.609, 5th percentile: 0.1933, 95th percentile: 2.3053
MEAN RMSE_VAL: 0.8309231561949189
MIN RMSE_VAL: 0.07904225984546721
MAX RMSE_VAL: 4.8738951485836495


## SDE

#Q-diffusion

In [ ]:
from numba import njit, prange

# =============================
# Global parameters
# =============================
n_traj = 50  # number of stochastic trajectories
sigma_results = {}
simulation_stats = {}  # directly stores mean and quantiles

# =============================
# Stochastic simulation function (direct statistics)
# =============================
@njit(parallel=True)
def simulate_SDE_stats(Q0, Pn, Perc, MU, LAMBDA, GAMMA, sigma, n_traj):
    N = len(Pn)
    QQ_sum = np.zeros(N)
    QQ_sq_sum = np.zeros(N)
    lower_acc = np.zeros(N)
    upper_acc = np.zeros(N)

    # Temporary storage for quantile computation
    temp_Q = np.zeros((N, n_traj))

    for i in prange(n_traj):
        S = np.zeros(N)
        S[0] = Q0
        for k in range(N - 1):
            drift = - (MU / LAMBDA) * S[k]**(2*MU - 1) + GAMMA * Perc[k+1] * Pn[k+1]
            diffusion = sigma * (S[k]) * np.random.normal(0.0, 1.0)
            S[k+1] = max(0, S[k] + drift + diffusion)
        temp_Q[:, i] = S
        QQ_sum += S

    # Mean and standard deviation
    mean_Q = QQ_sum / n_traj

    # 5% and 95% percentiles
    for t in range(N):
        sorted_vals = np.sort(temp_Q[t, :])
        lower_acc[t] = sorted_vals[int(0.05 * n_traj)]
        upper_acc[t] = sorted_vals[int(0.95 * n_traj)]

    return mean_Q, lower_acc, upper_acc

# =============================
# Loop over all stations
# =============================
for station_id, res in results.items():

    MU, LAMBDA, X1, GAMMA = res["params"]
    Q_obs = res["Q_obs"]
    Pn = np.nan_to_num(res["Pn"], nan=0.0)
    Perc = np.nan_to_num(res["Perc"], nan=0.0)

    N = len(Q_obs)
    b1 = int(N * b1_ratio)  # calibration

    # -----------------------------
    # Sigma estimation on calibration
    # -----------------------------
    num = 0.0
    den = 0.0
    for k in range(b1 - 1):
        if np.isnan(Q_obs[k]) or np.isnan(Q_obs[k+1]):
            continue
        drift = - (MU / LAMBDA) * Q_obs[k]**(2*MU - 1) + GAMMA * Perc[k+1] * Pn[k+1]
        incr = Q_obs[k+1] - Q_obs[k] - drift
        num += incr**2
        den += (Q_obs[k])**2

    sigma = np.sqrt(num / den)
    sigma_results[station_id] = sigma
    res["sigma"] = sigma
    print(f"Station {station_id}: sigma = {sigma:.4f}")

    # -----------------------------
    # Simulation over the entire period (statistics only)
    # -----------------------------
    mean_Q, lower, upper = simulate_SDE_stats(Q_obs[0], Pn, Perc, MU, LAMBDA, GAMMA, sigma, n_traj)

    # -----------------------------
    # Split calibration / validation
    # -----------------------------
    simulation_stats[station_id] = {
        "mean_calib": mean_Q[:b1],
        "lower_calib": lower[:b1],
        "upper_calib": upper[:b1],
        "mean_valid": mean_Q[b1:],
        "lower_valid": lower[b1:],
        "upper_valid": upper[b1:]
    }

Station A105003001: sigma = 0.5488
Station A107020001: sigma = 0.5083
Station A116003002: sigma = 0.5154
Station A140202001: sigma = 0.4137
Station A202030001: sigma = 0.2268
Station A211030001: sigma = 0.2421
Station A231020001: sigma = 0.2933
Station A234021001: sigma = 0.3396
Station A251020001: sigma = 0.2469
Station A273011002: sigma = 0.2854
Station A284020001: sigma = 0.3016
Station A330010001: sigma = 0.3463
Station A369011001: sigma = 0.2230
Station A373020001: sigma = 0.5040
Station A380020001: sigma = 0.1270
Station A402061001: sigma = 0.3904
Station A405062001: sigma = 0.3916
Station A417301001: sigma = 0.3419
Station A420063001: sigma = 0.3668
Station A433301001: sigma = 0.3013
Station A436203001: sigma = 0.2456
Station A443064001: sigma = 0.3340
Station A524201001: sigma = 0.4941
Station A543101001: sigma = 0.5697
Station A550061001: sigma = 0.3935
Station A605102001: sigma = 0.2864
Station A673122001: sigma = 0.4576
Station A712201001: sigma = 0.2820
Station A807101001: 

In [ ]:
# =========================
# Metric functions
# =========================
def NSE(obs, sim):
    if len(obs) == 0 or np.var(obs) == 0:
        return np.nan
    return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)

def RMSE(obs, sim):
    if len(obs) == 0:
        return np.nan
    return np.sqrt(np.mean((sim - obs)**2))

def interval_score(Q, L, U, alpha=0.1):
    IS = (U - L) + (2/alpha) * np.maximum(L - Q, 0) + (2/alpha) * np.maximum(Q - U, 0)
    return np.mean(IS)


# =========================
# Global lists
# =========================
NSE_cal_list, RMSE_cal_list, IS_cal_list = [], [], []
NSE_val_list, RMSE_val_list, IS_val_list = [], [], []


# =========================
# Loop over stations
# =========================
for station_id, res in results.items():

    Q_obs = res["Q_obs"]
    N = len(Q_obs)
    b1 = int(N * b1_ratio)

    stats = simulation_stats[station_id]

    # =========================
    # Internal function for Calibration or Validation
    # =========================
    def compute_metrics(Q_obs_part, mean_Q, lower, upper):
        valid_idx = (~np.isnan(Q_obs_part)) & (~np.isnan(lower)) & (~np.isnan(upper))

        Q = Q_obs_part[valid_idx]
        L = lower[valid_idx]
        U = upper[valid_idx]
        mean_Qv = mean_Q[valid_idx]

        if len(Q) == 0:
            return np.nan, np.nan, np.nan

        nse = NSE(Q, mean_Qv)
        rmse = RMSE(Q, mean_Qv)
        is_mean = interval_score(Q, L, U)

        return nse, rmse, is_mean

    # =========================
    # Calibration
    # =========================
    nse_cal, rmse_cal, is_cal = compute_metrics(
        Q_obs[:b1],
        stats["mean_calib"],
        stats["lower_calib"],
        stats["upper_calib"]
    )

    # =========================
    # Validation
    # =========================
    nse_val, rmse_val, is_val = compute_metrics(
        Q_obs[b1:],
        stats["mean_valid"],
        stats["lower_valid"],
        stats["upper_valid"]
    )

    # =========================
    # Storage
    # =========================
    if np.isfinite(nse_cal):
        NSE_cal_list.append(nse_cal)
        RMSE_cal_list.append(rmse_cal)
        IS_cal_list.append(is_cal)

    if np.isfinite(nse_val):
        NSE_val_list.append(nse_val)
        RMSE_val_list.append(rmse_val)
        IS_val_list.append(is_val)

    # =========================
    # Individual display
    # =========================
    print(f"Basin {station_id}:")
    #print(f"  --- Calibration ---")
    #print(f"    NSE  = {nse_cal:.3f}")
    #print(f"    RMSE = {rmse_cal:.3f}")
    #print(f"    IS   = {is_cal:.3f}")

    print(f"  --- Validation ---")
    print(f"    NSE  = {nse_val:.3f}")
    print(f"    RMSE = {rmse_val:.3f}")
    print(f"    IS   = {is_val:.3f}\n")

Basin A105003001:
  --- Validation ---
    NSE  = 0.612
    RMSE = 0.646
    IS   = 2.762

Basin A107020001:
  --- Validation ---
    NSE  = 0.577
    RMSE = 0.631
    IS   = 2.282

Basin A116003002:
  --- Validation ---
    NSE  = 0.675
    RMSE = 0.612
    IS   = 2.614

Basin A140202001:
  --- Validation ---
    NSE  = 0.618
    RMSE = 3.659
    IS   = 13.536

Basin A202030001:
  --- Validation ---
    NSE  = 0.793
    RMSE = 1.266
    IS   = 4.632

Basin A211030001:
  --- Validation ---
    NSE  = 0.694
    RMSE = 1.295
    IS   = 5.086

Basin A231020001:
  --- Validation ---
    NSE  = 0.821
    RMSE = 0.662
    IS   = 2.497

Basin A234021001:
  --- Validation ---
    NSE  = 0.692
    RMSE = 0.951
    IS   = 3.358

Basin A251020001:
  --- Validation ---
    NSE  = 0.817
    RMSE = 0.566
    IS   = 2.660

Basin A273011002:
  --- Validation ---
    NSE  = 0.794
    RMSE = 1.037
    IS   = 4.231

Basin A284020001:
  --- Validation ---
    NSE  = 0.696
    RMSE = 0.298
    IS   = 1.373

In [ ]:
print("\n================= STOCHASTIC PERFORMANCE =================\n")

# =========================
# Calibration
# =========================
print("----------- CALIBRATION -----------")
if NSE_cal_list:
    print(f"NSE  -> Median: {np.percentile(NSE_cal_list, 50):.3f}, "
          f"5th: {np.percentile(NSE_cal_list, 5):.3f}, "
          f"95th: {np.percentile(NSE_cal_list, 95):.3f}")
    print(f"Mean NSE: {np.mean(NSE_cal_list):.3f}, Min: {np.min(NSE_cal_list):.3f}, Max: {np.max(NSE_cal_list):.3f}")
    print(f"RMSE -> Median: {np.percentile(RMSE_cal_list, 50):.3f}, "
          f"5th: {np.percentile(RMSE_cal_list, 5):.3f}, "
          f"95th: {np.percentile(RMSE_cal_list, 95):.3f}")
    print(f"Mean RMSE: {np.mean(RMSE_cal_list):.3f}, Min: {np.min(RMSE_cal_list):.3f}, Max: {np.max(RMSE_cal_list):.3f}\n")
else:
    print("No valid CALIBRATION metrics available.\n")

# =========================
# Validation
# =========================
print("----------- VALIDATION -----------")
if NSE_val_list:
    print(f"NSE  -> Median: {np.percentile(NSE_val_list, 50):.3f}, "
          f"5th: {np.percentile(NSE_val_list, 5):.3f}, "
          f"95th: {np.percentile(NSE_val_list, 95):.3f}")
    print(f"Mean NSE: {np.mean(NSE_val_list):.3f}, Min: {np.min(NSE_val_list):.3f}, Max: {np.max(NSE_val_list):.3f}")
    print(f"RMSE -> Median: {np.percentile(RMSE_val_list, 50):.3f}, "
          f"5th: {np.percentile(RMSE_val_list, 5):.3f}, "
          f"95th: {np.percentile(RMSE_val_list, 95):.3f}")
    print(f"Mean RMSE: {np.mean(RMSE_val_list):.3f}, Min: {np.min(RMSE_val_list):.3f}, Max: {np.max(RMSE_val_list):.3f}\n")
else:
    print("No valid VALIDATION metrics available.\n")


================= STOCHASTIC PERFORMANCE =================

----------- CALIBRATION -----------
NSE  -> Median: 0.731, 5th: 0.479, 95th: 0.857
Mean NSE: 0.705, Min: -0.418, Max: 0.907
RMSE -> Median: 0.639, 5th: 0.192, 95th: 2.198
Mean RMSE: 0.842, Min: 0.079, Max: 5.545

----------- VALIDATION -----------
NSE  -> Median: 0.733, 5th: 0.453, 95th: 0.875
Mean NSE: 0.705, Min: -0.431, Max: 0.928
RMSE -> Median: 0.641, 5th: 0.208, 95th: 2.384
Mean RMSE: 0.871, Min: 0.078, Max: 5.253



In [ ]:
print("\n================= UNCERTAINTY PERFORMANCE =================\n")

# Calibration
print("----------- CALIBRATION -----------")
if IS_cal_list:
    print(f"Interval Score (90%) -> Median: {np.percentile(IS_cal_list, 50):.3f}, "
          f"5th: {np.percentile(IS_cal_list, 5):.3f}, "
          f"95th: {np.percentile(IS_cal_list, 95):.3f}")
    print(f"Mean IS: {np.mean(IS_cal_list):.3f}, Min: {np.min(IS_cal_list):.3f}, Max: {np.max(IS_cal_list):.3f}\n")
else:
    print("No valid CALIBRATION IS available.\n")

# Validation
print("----------- VALIDATION -----------")
if IS_val_list:
    print(f"Interval Score (90%) -> Median: {np.percentile(IS_val_list, 50):.3f}, "
          f"5th: {np.percentile(IS_val_list, 5):.3f}, "
          f"95th: {np.percentile(IS_val_list, 95):.3f}")
    print(f"Mean IS: {np.mean(IS_val_list):.3f}, Min: {np.min(IS_val_list):.3f}, Max: {np.max(IS_val_list):.3f}\n")
else:
    print("No valid VALIDATION IS available.\n")


================= UNCERTAINTY PERFORMANCE =================

----------- CALIBRATION -----------
Interval Score (90%) -> Median: 2.781, 5th: 0.957, 95th: 9.135
Mean IS: 3.488, Min: 0.317, Max: 19.905

----------- VALIDATION -----------
Interval Score (90%) -> Median: 2.735, 5th: 1.074, 95th: 9.245
Mean IS: 3.546, Min: 0.339, Max: 18.457

